![](./mira/gui/resources/pics/Logo_Sykno.svg "")

# __MiRa Data Evaluation__ - ___Jupyter Notebook___

---
### __Introduction__

**MiRa Data Measurement:**

- MiRa Measurements Path (HDF5 Files): `/mira_root/mira/meas/hdf5/`
- MiRa HDF5 Controller for interaction via MiRa Eval GUI or within the Jupyter Notebook: `/mira_root/mira/meas/mira_hdf5_ctrl.py` (see MiRa HDF5 Controller Usage)
- The MiRa radar data is a Numpy array `radar_data_cube` with a shape of `(n_frames, n_samples_per_chirp, n_rx, n_tx, n_shape_sets)` and its `dtype` is `float32`.


**MiRa HDF5 Structure:**
- __/__
  - __Data__
    - ***Frame_Data_Cube_XXXX_ZZZZ***
      - `@delta_time`: `np.float32`
      - `@shape`: `(n_samples_per_chirp, n_rx, n_tx, n_shapes)`
      - `@timestamp`: `np.float32`
      - ...
  - __Metadata__
    - ***mira_bgt_reg_content***
      - `@mira_bgt_reg_content`:
    - ***mira_bgt_reg_content_readable***
      - `@mira_bgt_reg_content_readable`:
    - ***mira_config***
      - `@mira_config`:
    - ***mira_radar_parameters*** 
      - `mira_radar_parameters`

***Frame_Data_Cube_XXXX_ZZZZ*** where ***XXXX*** are multiples of `4095` and ***ZZZZ*** is the frame counter with values in the range $[0, 4095]$.

- ***XXXX*** $\in \{k \cdot 4095 \mid k \in \mathbb{N}, 0 \leq k \leq 9999\}$
- ***ZZZZ*** $\in \{0, 1, 2, \ldots, 4095\}$


**Radar data cube structure:**
| Dimension            | Size |
|----------------------|------|
| Number of samples    | n_samples_per_chirp $\in \{2^n \mid n \in \mathbb{N}, 6 \leq n \leq 11\}$ |
| Number of RX antennas| n_rx_active_antennas $\in \{n \mid n \in \mathbb{N}, 1 \leq n \leq 4\}$ |
| Number of TX antennas| n_tx_active_antennas $\in \{n \mid n \in \mathbb{N}, 1 \leq n \leq 2\}$ |
| Number of shape sets | n_shapes $\in \{n \mid n \in \mathbb{N}, 1 \leq n \leq 4095\}$ |
| Number of frames     | n_frames $\in \{n \mid n \in \mathbb{N}, 0 \leq n \leq 4095\}$ |


**Chirp data structure:**

| Samples      | ADC Channel 1 (RX1) |ADC Channel 2 (RX2) |ADC Channel 3 (RX3) |ADC Channel 4 (RX4) |
|-----------------|------------------------------|-----------------|------------------------------|-----------------|
| 0         | dtype=np.uint16 | dtype=np.uint16 | dtype=np.uint16 | dtype=np.uint16 |
| 1         | 12 bit ADC | 12 bit ADC | 12 bit ADC | 12 bit ADC |
| 2         | 0 - 1.2 V | 0 - 1.2 V | 0 - 1.2 V | 0 - 1.2 V |
| ...             | ...                          | ...                          | ...                          | ...                          |
| n-1         | 0 - 4095 | 0 - 4095 | 0 - 4095 | 0 - 4095 |


**Working with HDF5:**

- List measurement structure:
```sh 
    h5ls -r mira_hdf5_filename.hdf5
```
- Inspect a specific group or dataset:
```sh
    h5dump -d /Data/Frame_Data_Cube_XXXX_ZZZZ mira_hdf5_filename.hdf5
```
- Inspect measurement meta data:
```sh
    h5dump -g /Metadata mira_hdf5_filename.hdf5
```
- Get statistics e.g. number of groups, datasets, and attributes:
```sh
    h5stat mira_hdf5_filename.hdf5
```


In [ ]:
description = "MiRa Root is: "
mira_root = !pwd
print(f"{description}{mira_root[0]}")
mira_meas_hdf5_path = f'{mira_root[0]}/mira/meas/hdf5/'
!tree -C {mira_meas_hdf5_path}

In [ ]:
mira_meas_hdf5_file_path = f'{mira_meas_hdf5_path}/MiRa6024_Measurement_05_03_2024_01_42_14_Default_Project_Default_Session.hdf5'

In [ ]:
!h5ls -r {mira_meas_hdf5_file_path}

In [ ]:
!h5dump -d /Data/Frame_Data_Cube_0000_0001 {mira_meas_hdf5_file_path}

In [ ]:
!h5dump -g /Metadata {mira_meas_hdf5_file_path}

In [ ]:
!h5stat {mira_meas_hdf5_file_path}

___
### __MiRa Data Evaluation__ - ___Python Code___


#### ***MiRa HDF5 Controller Usage***


In [ ]:
import os
import numpy as np
import ipywidgets as widgets
from mira.meas.mira_hdf5_ctrl import MIRA_HDF5_CTRL
from mira.rsys.mira_radar_sys import MIRA_RADAR_PARAMETER
mira_meas_hdf5_path = './mira/meas/hdf5/'

# List all files in the folder
file_names = os.listdir(mira_meas_hdf5_path)

# Filter out non-file entries if needed
file_names = [f for f in file_names if os.path.isfile(os.path.join(mira_meas_hdf5_path, f)) and f != '.gitkeep']
sorted_file_names = sorted(file_names)

# Dropdown for file selection
file_dropdown = widgets.Dropdown(
    options=file_names,
    description=f'Select a File:',
)
print((f'MiRa HDF5 Files in {mira_meas_hdf5_path}:'))
file_dropdown.layout.width = '650px'  # Adjust the width as needed
display(file_dropdown)



In [ ]:
from pathlib import Path
mira_hdf5_controller = MIRA_HDF5_CTRL(Path(mira_meas_hdf5_path+str(file_dropdown.value)).resolve())
data = mira_hdf5_controller.read_dataset("/Data/Frame_Data_Cube_0000_0023")
print(data)

In [ ]:
mira_hdf5_controller.print_hdf5_structure()

In [ ]:
stats = mira_hdf5_controller.get_dataset_statistics("/Data/Frame_Data_Cube_0000_0001")
print(stats)

In [ ]:
info = mira_hdf5_controller.get_dataset_info("/Data/Frame_Data_Cube_0000_0001")
print(info)

In [ ]:
frame_data_cube = mira_hdf5_controller.get_dataset_slice("/Data/Frame_Data_Cube_0000_0001")

In [ ]:
metadata = mira_hdf5_controller.read_metadata()
print(metadata)

In [ ]:
radar_param: MIRA_RADAR_PARAMETER = mira_hdf5_controller.unpickle_metadata()
print(radar_param)

In [ ]:
mira_data_cube = np.array(mira_hdf5_controller.load_radar_data_cube(), dtype=np.float32)
print(mira_data_cube.shape, mira_data_cube.dtype)

---
#### ***Inspect MiRa Dataset***

1. **Load python packages and MiRa HDF5 measurement file**


2. **Open MiRa HDF5 measurement file, load meta data and build radar data cube**

3. **MiRa HDF5 measurement - data profiling**

    __Inspect MiRa Radar System Parameters__
    - __radar_param__: meta-class contains __All__ other sub-classes
        - __sys__: sub-class contains __Radar__ specific values
        - __dsp__: sub-class contains __DSP__ parameters
        - __mon__: sub-class to __Monitor__ frame count, duration time, etc.
        - __gui__: sub-class contains __GUI__ relevant parameters
        - __meas__: sub-class contains __Measurement__ relevant parameters
        - __remt__: sub-class contains __Remote Control__ relevant parameters
        - __rply__: sub-class contains __Replay Control__ relevant parameters 

In [ ]:
print(f"{radar_param.__dict__=}")
print(f"{radar_param.dsp.__dict__=}")
print(f"{radar_param.gui.__dict__=}")
print(f"{radar_param.mon.__dict__=}")
print(f"{radar_param.sys.__dict__=}")
print(f"{radar_param.meas.__dict__=}")
print(f"{radar_param.remt.__dict__=}")
print(f"{radar_param.rply.__dict__=}")

3. **MiRa HDF5 measurement - data profiling**

    2. __Check MiRa Radar System Parameters__
    - __radar_param__: meta-class contains __All__ other sub-classes

In [ ]:
print(f"{mira_data_cube.shape=}")
print(f"{frame_data_cube.shape=}\n" + \
      f"{mira_hdf5_controller.delta_time=}\n" + \
      f"{mira_hdf5_controller.shape=}\n" + \
      f"{mira_hdf5_controller.timestamp=}")

print(f"{frame_data_cube[:1,:,:,0]}")


---
#### ***Postprocessing MiRa Data***

##### 1. **Load MiRa Processing Module and create a Data Processor**

In [ ]:
from mira.proc.mira_data_processing import MIRA_DATA_PROCESSOR
mira_data_processor = MIRA_DATA_PROCESSOR(radar_param)

mira_data_processor.padding_len = 0
mira_data_processor.data_preprocessor.init_window(window_func='hanning')
mira_data_processor.data_preprocessor.init_dsp_hp_filter(order=7, filter_type='sos', cutoff=100e3)


##### 2. **Time Singal Processing**

In [ ]:
scaled_raw_data = mira_data_processor.scale_raw_data(mira_data_cube[1,...], preprocessing=False, output_mean=False)
print(f"{scaled_raw_data.shape=}")
scaled_raw_data_mean = mira_data_processor.scale_raw_data(mira_data_cube[1,...], preprocessing=False, output_mean=True)
print(f"{scaled_raw_data_mean.shape=}")
scaled_raw_data_preprocessed = mira_data_processor.scale_raw_data(mira_data_cube[1,...], preprocessing=True, output_mean=False)
print(f"{scaled_raw_data_preprocessed.shape=}")
scaled_raw_data_preprocessed_mean = mira_data_processor.scale_raw_data(mira_data_cube[1,...], preprocessing=True, output_mean=True)
print(f"{scaled_raw_data_preprocessed_mean.shape=}")

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt

SHAPE_SET_NUMBER = 1

def plot_tx_rx_lines(data, plot_title_note: str='', 
                     x_label: str='Time (μs)', 
                     y_label: str='Voltage (mV)',
                     style: str='dark_background'):

    num_tx = data.shape[2]
    num_rx = data.shape[1]
    x_max =radar_param.sys.ramp_time[0]*1e6
    x_values = np.linspace(0, x_max, scaled_raw_data.shape[0])
    with plt.style.context(style):
        for tx in range(num_tx):
            # Initialize the plot for each transmitter
            plt.figure(figsize=(10, 6))
            
            # Iterate over the receivers to plot each as a separate line
            for rx in range(num_rx):
                plt.plot(x_values, data[:, rx, tx], label=f'TX{tx+1}_RX{rx+1}')
            
            # Adding labels and title
            plt.xlabel(x_label)
            plt.ylabel(y_label)
            plt.title(f'Plot of TX{tx+1} with {num_rx} RX Lines {plot_title_note}')
            
            # Adding legend
            plt.legend()
            
            # Show the plot
            plt.show()

plot_tx_rx_lines(scaled_raw_data[...,-1], 'Raw Data No Mean')
plot_tx_rx_lines(scaled_raw_data_mean, 'Raw Data Mean')
plot_tx_rx_lines(scaled_raw_data_preprocessed[...,-1], 'Raw Data Preprocessed No Mean')
plot_tx_rx_lines(scaled_raw_data_preprocessed_mean, 'Raw Data Preprocessed Mean')

In [ ]:
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm
from matplotlib import animation
from IPython.display import HTML

# Generate dummy data for demonstration
# scaled_raw_data_preprocessed = np.random.rand(64, 4, 2, 512)  # [samples, rx, tx, frames]
mpl.rcParams['animation.embed_limit'] = 10000
# Extract dimensions
num_samples, num_rx, num_tx, num_frames = scaled_raw_data.shape

# Prepare the figure and axes
fig, axs = plt.subplots(num_tx, figsize=(10, 6 * num_tx), sharex=True, constrained_layout=True)
plt.style.use('dark_background')

if num_tx == 1:
    axs = [axs]  # Make it iterable for a single transmitter case

# Initialize lines for each rx in tx plots and set initial data
lines = []
for tx in range(num_tx):
    for rx in range(num_rx):
        line, = axs[tx].plot(np.arange(num_samples), scaled_raw_data[:, rx, tx, 0], label=f'RX{rx+1}')
        lines.append(line)
    axs[tx].set_title(f'TX{tx+1}')
    axs[tx].legend(loc='upper right')
    axs[tx].set_xlabel('Samples')
    axs[tx].set_ylabel('Amplitude')
    axs[tx].set_xlim(0, num_samples-1)
    axs[tx].set_ylim(np.min(scaled_raw_data), np.max(scaled_raw_data))

# Define animation update function
def update(frame):
    idx = 0
    for tx in range(num_tx):
        for rx in range(num_rx):
            lines[idx].set_ydata(scaled_raw_data[:, rx, tx, frame])
            idx += 1
    return lines


# Modify the update function call within FuncAnimation
ani = animation.FuncAnimation(fig, update, frames=[i for i in tqdm(range(num_frames))], interval=100)


plt.close(fig)
# Display the animation
HTML(ani.to_jshtml())


In [ ]:
import numpy as np

# Function to check and overwrite slices
def clear_data(arr):
    frame_correction_dict = {
        'counter': 0,
        'position': [],
    }
    # Iterate over dimensions 0, 2, 3, and 4
    for frame_index in range(1, arr.shape[0], 1):  # dim 0
        for rx_index in range(arr.shape[2]):  # dim 2
            for tx_index in range(arr.shape[3]):  # dim 3
                for shape_index in range(1, arr.shape[4]):  # dim 4, start from 1 to be able to refer to the previous slice
                    # Check if all elements along dim 1 are zeros for the given slice
                    if np.all(arr[frame_index, :, rx_index, tx_index, shape_index] == 0):
                        # Overwrite with the values from the previous slice in dim 4
                        # print(arr[i, :, j, k, l])
                        arr[frame_index, :, rx_index, tx_index, shape_index] = arr[frame_index, :, rx_index, tx_index, shape_index-1]
                        frame_correction_dict['position'].append((shape_index,rx_index,tx_index,shape_index))
                        frame_correction_dict['counter'] += 1
                        # print('clear up', f"{frame_index=}, {rx_index=}, {tx_index=}, {shape_index=}, {frame_correction_dict['counter']}")
    return frame_correction_dict

# Apply the function to your array
frame_correction_result = clear_data(mira_data_cube)
print(frame_correction_result)



##### 3. **Spectrum Singal Processing**

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from mpl_toolkits.mplot3d import Axes3D
from matplotlib import animation
from IPython.display import HTML
c = 0

for i in range(mira_data_cube.shape[0]):
    for ii in range(mira_data_cube.shape[4]):
        is_zero = np.any(np.all(mira_data_cube[i, :, 0, 0, ii] == 0))
        if is_zero:
            c+=1
            
spectrum = mira_data_processor._calc_rfft_channels(mira_data_cube.transpose((1,2,3,4,0)))
subset = spectrum[:, 0, 0, :, :]  
subset = subset.transpose(1,0,2)

# Create meshgrid for the x and y dimensions
x = np.arange(0, (radar_param.sys.frame_duration * 1e3), (radar_param.sys.frame_duration * 1e3)/subset.shape[1])  # Note the change to match 'subset' dimensions
y = np.arange(0, radar_param.sys.max_range, radar_param.sys.max_range/subset.shape[0])
x, y = np.meshgrid(x, y)

# Initialize 3D plot
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection='3d')

# Initial plot
surf = ax.plot_surface(x, y, subset[:, :, 0], cmap='viridis')  # Initial Z-data

# Update function for animation
def update(frame):
    ax.clear()
    ax.plot_surface(x, y, subset[:, :, frame], cmap='viridis')
    ax.set_xlabel('Range (m)')
    ax.set_ylabel('Time (ms)')
    ax.set_zlabel('Magnetude (dB)')
    ax.set_title('3D Waterfall spectrogram')

    # Optionally, set the same zlim for each frame to have consistent scaling
    ax.set_zlim(-20, 40)

# Create animation
ani = animation.FuncAnimation(fig, update, frames=subset.shape[2], interval=100)
ani.save('./mira/meas/animation/animation.mp4', writer='ffmpeg', fps=10)
# Close the plot to prevent it from displaying statically
plt.close()

# Display the animation
# HTML(ani.to_jshtml(fps=10))


##### 4. **Spectrogram Signal Processing**


In [ ]:
print(f"{mira_data_cube.shape=}")
for frame in range(mira_data_cube.shape[0]-1):
    spectrogram_map_1 = mira_data_processor._calc_spectogram(mira_data_cube[frame+1,:,:,0,:])
mira_data_processor.spectogram_map = np.zeros((1,1,1), dtype=np.float32)
for frame in range(mira_data_cube.shape[0]-1):
    spectrogram_map_2 = mira_data_processor._calc_spectogram(mira_data_cube[frame+1,:,:,1,:])
spectrogram_map = np.stack((spectrogram_map_1, spectrogram_map_2), axis=-1)
del spectrogram_map_1, spectrogram_map_2
print(f"{spectrogram_map.shape=}")

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np

# Assuming 'radar_param' and 'spectrogram_map' are defined in your context
# And assuming 'spectrogram_map' is your numpy array with shape (60, 128, 4)

# Titles for each subplot
titles = ['RX1', 'RX2', 'RX3', 'RX4', 'RX1', 'RX2', 'RX3', 'RX4']
x_min, x_max = 0, radar_param.sys.max_range  # microseconds
y_min, y_max = 0, radar_param.sys.frame_duration * 1e3  # milliseconds

# Calculate the extent for imshow
extent = [x_min, x_max, y_min, y_max]

with plt.style.context('dark_background'):
    # Set up the figure and axes for 8 subplots
    fig, axs = plt.subplots(8, 1, figsize=(15, 30))  # Modified the figsize for better visibility
    
    for tx in range(spectrogram_map.shape[3]):  # Assuming this is 2 for TX1 and TX2
        for rx in range(spectrogram_map.shape[2]):  # Assuming this is 4 for RX1 to RX4
            dataset = spectrogram_map[:, :, rx, tx]
            
            # Calculate the correct index for the subplot
            index = tx * 4 + rx
            # Plot the dataset
            im = axs[index].imshow(dataset, cmap='jet', aspect='auto', extent=extent, origin='lower')

            # Set the title for the subplot
            axs[index].set_title(f"TX{tx+1} RX{rx+1}", color='white')
            
            # Add a colorbar to each subplot
            cbar = fig.colorbar(im, ax=axs[index])
            cbar.ax.yaxis.set_tick_params(color='white')
            cbar.outline.set_edgecolor('white')
            for text in cbar.ax.yaxis.get_ticklabels():
                text.set_color('white')

            # Set axis labels
            axs[index].set_xlabel('Range (m)')
            axs[index].set_ylabel('Time (ms)')

# Adjust layout to prevent overlap
plt.tight_layout()
plt.show()

In [ ]:
# TODO Implement some interactive features to jupyter notebook
import ipywidgets as widgets
from IPython.display import display
import datetime
import matplotlib.pyplot as plt
import numpy as np

# Example datasets
datasets = {
    'Sine Wave': np.sin,
    'Cosine Wave': np.cos
}

# Dropdown for dataset selection
dataset_dropdown = widgets.Dropdown(
    options=list(datasets.keys()),
    value='Sine Wave',
    description='Dataset:',
)

# Slider for frequency adjustment
frequency_slider = widgets.FloatSlider(
    value=1.0,
    min=0.1,
    max=5.0,
    step=0.1,
    description='Frequency:',
)

# Function to update plot
def update_plot(dataset_name, frequency):
    plt.clf() # Clear the current figure
    func = datasets[dataset_name] # Get the function to use
    x = np.linspace(0, 2 * np.pi, 1000)
    y = func(frequency * x)
    plt.plot(x, y)
    plt.show()

# Interactive widget
widgets.interactive_output(update_plot, {'dataset_name': dataset_dropdown, 'frequency': frequency_slider})

# Display widgets
display(dataset_dropdown, frequency_slider)

# Button widget
button = widgets.Button(description="Click Me!")
def on_button_clicked(b):
    print("Button clicked.")

button.on_click(on_button_clicked)

# Accordion widget
accordion = widgets.Accordion(children=[widgets.IntSlider(), widgets.Text()])
accordion.set_title(0, 'Slider')
accordion.set_title(1, 'Text')

display(button, accordion)
